Откуда берутся датасеты? Практический проект по сбору данных и работе с текстами

Цель:
собрать и подготовить датасет с использованием методов парсинга веб-страниц.
построить и оценить модель машинного обучения на основе собранных данных.


Описание/Пошаговая инструкция выполнения домашнего задания:
Часть 1. Парсинг.

По аналогии с занятием, возьмите интересующий вас сайт, на котором можно пособирать какие-то данные (и при этом API не предоставляется).


Идеальный датасет должен иметь текстовое описание некоторого объекта и некоторую целевую переменную, соответствующую этому объекту. Например:

Сайт новостей: текстовое описание - сама новость, целевая переменная - количество просмотров новости (можно поделить на число дней с момента даты публикации, чтобы получить “среднее число просмотров в день”).
Сайт с товарами/книгами/фильмами: текстовое описание товара/книги/фильма + средний рейтинг в качестве целевой переменной.
Блоги - тексты заметок + число просмотров.
И любые другие ваши идеи, которые подходят под такой формат.

Напишите свой парсер, который будет бегать по страничкам и автоматически что-то собирать.


Не забывайте, что парсинг - это ответственное мероприятие, поэтому не бомбардируйте несчастные сайты слишком частыми запросами (можно ограничить число запросов в секунду при помощи time.sleep(0.3), вставленного в теле цикла)


Часть 2. NLP.

Разбейте собранные данные на train/test, отложив 20-30% наблюдений для тестирования.
Примените tf-idf преобразование для текстового описания. Используйте как отдельные токены, так и биграммы, отсейте стоп-слова, а также слова, которые встречаются слишком редко или слишком часто (параметры min/max_df), не забудьте убрать l2 регуляризацию, которая по умолчанию включена.
Если в вашем датасете целевая переменная непрерывная (например, среднее число просмотров в день), то воспользуйтесь линейной регрессией, если дискретная (положительный/отрицательный отзыв), то логистической.
Постройте регрессию с настройкой параметра регуляризации, оцените качество при помощи соответствующих задаче метрик.
Визуализируйте получившиеся коэффициенты регрессии (возьмите топ-50 слов)
Проинтерпретируйте результаты.

P.S. Если с парсингом не задалось или данных собралось слишком мало - не отчаивайтесь, главное, что ваш парсер собрал хоть что-то! А для второй части задания можно скачать данные по отзывам на фильмы с сайта IMDB (https://www.kaggle.com/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews), в которых для каждого отзыва поставлена семантическая оценка - "позитивный" или "негативный".


Критерии оценки:
Получилось написать работающий парсер - 4 балла.
Предобработка текста при помощи tf-idf - 2 балла.
Построение моделей и визуализация результатов - 4 балла.

## Часть 1. Парсинг


Пробую парсинг с habr.com разделы статей про мл и ии

Выбрал библиотеки **requests и BeautifulSoup**.

В самих страницах есть url, title, автор, дата публикации, тэги и статистика (лайки, просмотры, добавка в избр ..)

In [2]:
# =====================
# 1. Setup
# =====================
import os
import re
import time
import json
from datetime import datetime

import requests
from bs4 import BeautifulSoup

import numpy as np
import pandas as pd
import plotly.express as px

RANDOM_STATE = 555
pd.set_option('display.max_columns', None)

base_dir = os.getcwd()
input_dir = os.path.join(base_dir, 'csv_input')
output_dir = os.path.join(base_dir, 'csv_output')
os.makedirs(output_dir, exist_ok=True)

# ---- habr.com parser config ----
N_ARTICLES = 500        # how many articles to collect 
REQUEST_DELAY = 0.3     # between requests - secs
PARSE_ENABLED = False    # Set False - skip parsing function and load the previously saved CSV

HABR_BASE = 'https://habr.com'
TOPICS = [
    'https://habr.com/ru/hubs/machine_learning/articles/',
    'https://habr.com/ru/hubs/artificial_intelligence/articles/',
]
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
}

habr_csv_path = os.path.join(output_dir, 'habr_ml_articles.csv')
imdb_csv_path = os.path.join(input_dir, 'IMDB_dataset.csv')
results_csv_path = os.path.join(output_dir, 'model_results.csv')

print('N_ARTICLES:', N_ARTICLES)
print('REQUEST_DELAY:', REQUEST_DELAY, 'seconds')
print('PARSE_ENABLED:', PARSE_ENABLED)
print('habr output:', habr_csv_path)
print('imdb input:', imdb_csv_path)

N_ARTICLES: 500
REQUEST_DELAY: 0.3 seconds
PARSE_ENABLED: False
habr output: E:\py_otus\ml-pro\homework_03_23\csv_output\habr_ml_articles.csv
imdb input: E:\py_otus\ml-pro\homework_03_23\csv_input\IMDB_dataset.csv


In [3]:
#--------------------
# 2. Article parser
#--------------------
# Trying requests + BeautifulSoup libs to fetch articles
# and we fetch likes and comments from json inside the article page

ARTICLE_LINK_RE = re.compile(r'^/ru/(?:articles|companies/[^/]+/articles)/(\d+)/?$')
PINIA_MARKER = 'window.__PINIA_STATE__='


def get_soup(url, session):
    response = session.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def collect_article_links(n_articles):
    # move through pages 1, 2, .. to get unique article ids
    session = requests.Session()
    links = {}
    for hub_url in TOPICS:
        page = 1
        while len(links) < n_articles:
            url = hub_url if page == 1 else hub_url + 'page' + str(page) + '/'
            try:
                soup = get_soup(url, session)
            except requests.RequestException as error:
                print('list page failed:', url, type(error).__name__)
                break
            new_count = 0
            for tag in soup.find_all('a', href=True):
                match = ARTICLE_LINK_RE.match(tag['href'])
                if match and match.group(1) not in links:
                    links[match.group(1)] = HABR_BASE + '/ru/articles/' + match.group(1) + '/'
                    new_count += 1
            print('list page', url, '| new links:', new_count, '| total:', len(links))
            if new_count == 0:
                break
            page += 1
            time.sleep(REQUEST_DELAY)
    return list(links.values())[:n_articles]


def extract_pinia_article(page_html):
    # pull the current article dict out of the embedded JSON state
    start = page_html.find(PINIA_MARKER)
    if start == -1:
        return {}
    state, _ = json.JSONDecoder().raw_decode(page_html[start + len(PINIA_MARKER):])
    articles = state.get('articlesList', {}).get('articlesList', {})
    return next(iter(articles.values()), {})


def parse_article(url, session):
    # title, date and body text come from the object model,
    # counters come from the embedded PINIA state
    response = session.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')

    title_tag = soup.find('h1')
    time_tag = soup.find('time')
    body_tag = soup.find('div', id='post-content-body')

    article = extract_pinia_article(response.text)
    stats = article.get('statistics', {})

    return {
        'url': url,
        'title': title_tag.get_text(' ', strip=True) if title_tag else None,
        'published': time_tag.get('datetime') if time_tag else None,
        'author': (article.get('author') or {}).get('alias'),
        'hubs': '; '.join(hub.get('title', '') for hub in (article.get('hubs') or [])),
        'text': body_tag.get_text(' ', strip=True) if body_tag else None,
        'score': stats.get('score'),
        'votes': stats.get('votesCount'),
        'views': stats.get('readingCount'),
        'comments': stats.get('commentsCount'),
        'favorites': stats.get('favoritesCount'),
    }


def parse_habr_articles(n_articles):
    session = requests.Session()
    urls = collect_article_links(n_articles)
    print('unique article urls to parse:', len(urls))
    records = []
    for index, url in enumerate(urls, start=1):
        try:
            records.append(parse_article(url, session))
        except requests.RequestException as error:
            print('article failed:', url, type(error).__name__)
        if index % 25 == 0 or index == len(urls):
            print('parsed', index, 'of', len(urls))
        time.sleep(REQUEST_DELAY)
    return pd.DataFrame(records)

In [4]:
#--------------------
# 3. Run the parser (or load the saved CSV) and save the dataset
#--------------------
if PARSE_ENABLED:
    df_habr = parse_habr_articles(N_ARTICLES)
    df_habr.to_csv(habr_csv_path, index=False, encoding='utf-8-sig')
    print('saved', len(df_habr), 'articles to', habr_csv_path)
elif os.path.exists(habr_csv_path):
    df_habr = pd.read_csv(habr_csv_path, encoding='utf-8-sig')
    print('loaded', len(df_habr), 'articles from', habr_csv_path)
else:
    df_habr = pd.DataFrame()
    print('no saved CSV found, set PARSE_ENABLED = True to parse data')

df_habr.head()

loaded 500 articles from E:\py_otus\ml-pro\homework_03_23\csv_output\habr_ml_articles.csv


,url,title,published,author,hubs,text,score,votes,views,comments,favorites
0,https://habr.com/ru/articles/1110754/,15 субагентов Claude Code писали систему. Прин...,2026-11-21T19:06:01.000Z,Aflubin1978,Управление разработкой; Программирование; Тест...,"Субагент сдаёт волну: отчёт на страницу, тесты...",-1,1,195,0,2
1,https://habr.com/ru/articles/1110692/,"Gemini 4 Neon догнала Opus 5.5, но релиз почти...",2026-11-21T13:49:12.000Z,FactChain,Искусственный интеллект; Машинное обучение; Пр...,Еще неделю назад я бы рассмеялся в лицо любому...,6,7,1782,11,10
2,https://habr.com/ru/articles/1106704/,Дообучаем на Mac малую языковую модель для RAG...,2026-11-21T13:12:10.000Z,kmoseenk,Блог компании OTUS; Машинное обучение; Искусст...,Недавно мы попробовали обобщать результаты сем...,4,3,443,0,10
3,https://habr.com/ru/articles/1110660/,Зеркальная аугментация лидара: почему при боль...,2026-11-21T12:26:15.000Z,Natali397,Алгоритмы; Машинное обучение; Робототехника; Ф...,При проектировании систем автономной навигации...,1,3,215,0,3
4,https://habr.com/ru/articles/1110646/,Стартап учит нейросети разбираться в запахах и...,2026-11-21T11:57:04.000Z,maybe_elf,Искусственный интеллект; Машинное обучение; Ра...,Стартап Osmo создаёт первую в мире «базовую мо...,3,1,144,0,1


In [5]:
#--------------------
# 4. EDA of the parsed articles
#--------------------
# target variable is the article score (net votes), views_per_day  - as option
if len(df_habr) > 0:
    print('shape:', df_habr.shape)
    print('nulls per column:')
    print(df_habr.isnull().sum())
    print()

    df_habr['text_len'] = df_habr['text'].fillna('').astype(str).str.len()
    now_utc = pd.Timestamp.now(tz='UTC')
    published_dt = pd.to_datetime(df_habr['published'])
    df_habr['age_days'] = (now_utc - published_dt).dt.total_seconds() / 86400
    df_habr['views_per_day'] = df_habr['views'] / df_habr['age_days'].clip(lower=1.0)
    print(df_habr[['score', 'votes', 'views', 'comments', 'favorites', 'text_len', 'views_per_day']]
          .describe().round(2))
    print()
    print('sample article text (first 255 chars):')
    print(df_habr['text'].iloc[0][:255])

    score_values = df_habr['score'].dropna()
    hist_counts, _ = np.histogram(score_values, bins=40)
    top = int(np.ceil(hist_counts.max() / 10) * 10)
    fig = px.histogram(score_values, nbins=40, title='Habr ML/AI articles: score distribution')
    fig.update_layout(showlegend=False)
    fig.update_yaxes(range=[0, top])
    fig.show()
else:
    print('df_habr is empty')

shape: (500, 11)
nulls per column:
url          0
title        0
published    0
author       0
hubs         0
text         0
score        0
votes        0
views        0
comments     0
favorites    0
dtype: int64

        score   votes     views  comments  favorites   text_len  views_per_day
count  500.00  500.00    500.00    500.00     500.00     500.00         500.00
mean     9.92    7.96   2719.15      8.05      12.51   13115.18         191.30
std     17.83   14.55   5698.22     28.75      30.11   13155.76         486.55
min    -15.00    0.00    125.00      0.00       0.00     301.00           3.85
25%      1.00    2.00    485.00      0.00       3.00    3958.50          30.37
50%      5.00    4.00    945.00      2.00       5.00   10157.00          65.65
75%     11.00    8.00   2489.25      6.00      12.00   18199.25         170.47
max    215.00  180.00  60720.00    400.00     513.00  131569.00        8128.78

sample article text (first 255 chars):
Субагент сдаёт волну: отчёт на стра

## Часть 2. NLP: классификация отзывов IMDB

Датасет IMDB (50k movie reviews, Kaggle): текст отзыва + метка `positive`/`negative` -
дискретная целевая переменная, поэтому по заданию используем логистическую регрессию.

Пайплайн: очистка текста -> train/test 70/30 -> TF-IDF (уни- и биграммы, стоп-слова,
min/max_df, без l2-нормализации) -> LogisticRegression с настройкой C -> метрики,
топ-50 коэффициентов, интерпретация.

In [5]:
#--------------------
# 5. IMDB dataset: load and clean
#--------------------
all_cols = pd.read_csv(imdb_csv_path, nrows=0).columns.tolist()
print('columns:', all_cols)
exclude_cols = []  # both columns are needed
use_cols = [col for col in all_cols if col not in exclude_cols]

df_imdb = pd.read_csv(imdb_csv_path, usecols=use_cols)
print('raw shape:', df_imdb.shape)

# remove html tags like <br /> that IMDB reviews contain, collapse whitespace
tag_re = re.compile(r'<[^>]+>')
df_imdb['review'] = (
    df_imdb['review']
    .str.replace(tag_re, ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# drop duplicated reviews and empty rows
print('duplicated reviews:', df_imdb.duplicated(subset='review').sum())
df_imdb = df_imdb.drop_duplicates(subset='review').dropna().reset_index(drop=True)

# binary target: positive = 1, negative = 0
df_imdb['label'] = (df_imdb['sentiment'] == 'positive').astype(int)
print('clean shape:', df_imdb.shape)
print(df_imdb['sentiment'].value_counts())
df_imdb.head()

columns: ['review', 'sentiment']


raw shape: (50000, 2)


duplicated reviews: 422
clean shape: (49578, 3)
sentiment
positive    24882
negative    24696
Name: count, dtype: int64


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. The filming tec...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [6]:
#--------------------
# 6. IMDB EDA: class balance and review lengths
#--------------------
balance = df_imdb['sentiment'].value_counts().reset_index()
balance_top = int(np.ceil(balance['count'].max() / 1000) * 1000)
fig = px.bar(balance, x='sentiment', y='count', title='IMDB: class balance')
fig.update_yaxes(range=[0, balance_top])
fig.show()

df_imdb['review_len'] = df_imdb['review'].str.len()
len_counts, _ = np.histogram(df_imdb['review_len'], bins=50)
len_top = int(np.ceil(len_counts.max() / 1000) * 1000)
fig = px.histogram(df_imdb, x='review_len', nbins=50, title='IMDB: review length distribution')
fig.update_yaxes(range=[0, len_top])
fig.show()

print(df_imdb['review_len'].describe().round(1))

count    49578.0
mean      1288.1
std        973.5
min         32.0
25%        690.0
50%        956.0
75%       1562.0
max      13593.0
Name: review_len, dtype: float64


In [7]:
#--------------------
# 7. Train/test split and TF-IDF vectors
#--------------------

#
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

X = df_imdb['review']
y = df_imdb['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE,
)
print('train:', X_train.shape[0], '| test:', X_test.shape[0])

# 1 word, 2 words (unigrams + bigrams), english stopwords, rare/frequent token filtering,
# norm=None disables the default l2 normalization as said in the task
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words='english',
    min_df=10,
    max_df=0.7,
    norm=None,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print('tf-idf matrix:', X_train_tfidf.shape)
print('vocabulary size:', len(tfidf.vocabulary_))
print('non-zero density: {:.4f}'.format(density))

train: 34704 | test: 14874


tf-idf matrix: (34704, 48500)
vocabulary size: 48500
non-zero density: 0.0022


In [8]:
#--------------------
# 8. Logistic regression with regularization
#--------------------
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)
grid.fit(X_train_tfidf, y_train)
print('cv results:')
for mean_score, params in zip(grid.cv_results_['mean_test_score'], grid.cv_results_['params']):
    print('  C = {:>6} | cv accuracy: {:.4f}'.format(params['C'], mean_score))
print('best C:', grid.best_params_['C'], '| best cv accuracy: {:.4f}'.format(grid.best_score_))

best_model = grid.best_estimator_

for name, X_set, y_set in [('train', X_train_tfidf, y_train), ('test', X_test_tfidf, y_test)]:
    pred = best_model.predict(X_set)
    proba = best_model.predict_proba(X_set)[:, 1]
    print()
    print(name, 'accuracy: {:.4f} | roc-auc: {:.4f}'.format(
        accuracy_score(y_set, pred), roc_auc_score(y_set, proba)))
    print(classification_report(y_set, pred, target_names=['negative', 'positive']))

cm = confusion_matrix(y_test, best_model.predict(X_test_tfidf))
fig = px.imshow(
    cm, text_auto=True,
    x=['negative', 'positive'], y=['negative', 'positive'],
    title='IMDB test: confusion matrix',
)
fig.update_xaxes(title='predicted')
fig.update_yaxes(title='actual')
fig.show()

cv results:
  C =   0.01 | cv accuracy: 0.8902
  C =    0.1 | cv accuracy: 0.8842
  C =      1 | cv accuracy: 0.8863
  C =     10 | cv accuracy: 0.8841
  C =    100 | cv accuracy: 0.8836
best C: 0.01 | best cv accuracy: 0.8902

train accuracy: 0.9994 | roc-auc: 1.0000
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     17287
    positive       1.00      1.00      1.00     17417

    accuracy                           1.00     34704
   macro avg       1.00      1.00      1.00     34704
weighted avg       1.00      1.00      1.00     34704


test accuracy: 0.8922 | roc-auc: 0.9550
              precision    recall  f1-score   support

    negative       0.90      0.89      0.89      7409
    positive       0.89      0.90      0.89      7465

    accuracy                           0.89     14874
   macro avg       0.89      0.89      0.89     14874
weighted avg       0.89      0.89      0.89     14874



In [9]:
#--------------------
# 8b. Log metrics to model_results.csv
#--------------------
test_pred = best_model.predict(X_test_tfidf)
test_proba = best_model.predict_proba(X_test_tfidf)[:, 1]

new_row = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'LogisticRegression (IMDB tf-idf)',
    'hyperparameters': 'C={}, ngram=(1,2), min_df=10, max_df=0.7, norm=None'.format(grid.best_params_['C']),
    'accuracy': round(accuracy_score(y_test, test_pred), 4),
    'precision': round(precision_score(y_test, test_pred), 4),
    'recall': round(recall_score(y_test, test_pred), 4),
    'f1': round(f1_score(y_test, test_pred), 4),
    'roc_auc': round(roc_auc_score(y_test, test_proba), 4),
}
results_df = pd.DataFrame([new_row])
if os.path.exists(results_csv_path):
    results_df = pd.concat([pd.read_csv(results_csv_path), results_df], ignore_index=True)
results_df.to_csv(results_csv_path, index=False)

results_df = results_df.drop_duplicates(subset='model', keep='last').reset_index(drop=True)
print(results_df.to_string(index=False))

          timestamp                            model                                       hyperparameters  accuracy  precision  recall     f1  roc_auc
2026-11-21 23:25:30 LogisticRegression (IMDB tf-idf) C=0.01, ngram=(1,2), min_df=10, max_df=0.7, norm=None    0.8922     0.8875  0.8991 0.8933    0.955


In [10]:
#--------------------
# 9. Top-50 regression coeff (25 positive, 25 negative)
#--------------------
feature_names = np.array(tfidf.get_feature_names_out())
coefs = best_model.coef_[0]

top_pos_idx = np.argsort(coefs)[::-1][:25]
top_neg_idx = np.argsort(coefs)[:25]

print('top-25 positive (push review towards positive):')
print(', '.join(feature_names[top_pos_idx]))
print()
print('top-25 negative (push review towards negative):')
print(', '.join(feature_names[top_neg_idx]))


def plot_coef_bar(indices, title):
    frame = pd.DataFrame({
        'feature': feature_names[indices],
        'coefficient': coefs[indices],
    }).iloc[::-1]
    fig = px.bar(frame, x='coefficient', y='feature', orientation='h', title=title)
    if frame['coefficient'].min() >= 0:
        fig.update_xaxes(range=[0, np.ceil(frame['coefficient'].max() * 1.05)])
    else:
        fig.update_xaxes(range=[np.floor(frame['coefficient'].min() * 1.05), 0])
    fig.update_layout(height=650)
    fig.show()


plot_coef_bar(top_pos_idx, 'Top-25 positive coefficients (positive sentiment)')
plot_coef_bar(top_neg_idx, 'Top-25 negative coefficients (negative sentiment)')

top-25 positive (push review towards positive):
great, excellent, best, funniest, perfect, wonderful, brilliant, 10 10, amazing, superb, enjoyable, favorite, today, refreshing, fantastic, hilarious, loved, fun, perfectly, enjoyed, incredible, enjoy, highly recommend, entertaining, loved movie

top-25 negative (push review towards negative):
worst, awful, waste, boring, disappointment, disappointing, poorly, bad, terrible, avoid, horrible, poor, dull, worse, fails, lame, forgettable, annoying, mediocre, unfunny, save, supposed, laughable, waste time, lacks


In [6]:
# =====================
# 10. Conclusions
# =====================
#
# Для парсинга взял хабр. конкретно темы про мл и ии. статей оказалось прямо вообще много. для выборки взял 500шт свежих. плюс там есть комменты, лайки .. это тоже подтянулось из jsonов в страницах.
# Думал потом по этим статьям делать след задание, взял текст статей. потом передумал. с отзывами про фильмы оказалось проще.
#
# Одна статья попалась с -15 голосами (5+ и 20-) https://habr.com/ru/articles/1107848/
# Специально пошел проверять из интереса. 25 раз добавили в избранное. 28 комментов и по тексту не сильно негативные отзывы. больше 10тыс просмотров было.  статью не читал :)
# итоги по статистике в разделе 4. EDA  и в csv_output\habr_ml_articles.csv
#
# Получилось здорово, буду пользоваться. (если с другими сайтами будет так же просто)
#
#
# Для второй части взял отзывы по фильмам по ссылке. размер таблицы 64,5 мб. в гит не включал.
# Были дубликаты. по итогу получилось 49 578 строк отзывов. причем практически пополам позитив и негатив
#
# после отсева мин в 10 отзывах и макс в 70% отзывов, в итоге получилось 48.5 тыс фич с плотностью матрицы 0.0022
# 21461 однословных и 27039 двусловных

# самые топ 50 параметров для позитивных: great, excellent, best, wonderful, brilliant, highly recommend для негативных worst, awful, waste, boring, disappointing, terrible ..
# полный список топ 50 чуть выше.
#
# Думал такое же сделать с комментами на хабре, но тут смысл разный и не очень понятно как комменты отмаркировать - отзывы легче отметить как позитив или негатив. а комменты там есть вообще непонятные.  поэтому и взял отзывы чтобы задание получилось.
# И метрики неплохо получились по обучению тестах на фильмах
